# 04 — Train: Ridge regression

Searches Ridge regularization for the configured target station's direct 24-hour water-level forecast over the joined feature artifacts, then evaluates the selected model once on the sealed test cohort.

**Inputs:** joined train/test feature artifacts and their metadata contract  
**Outputs:** in-notebook prediction preview/test metrics, an MLflow run hierarchy, and the selected model plus manifest in `models/`

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and loads the joined feature metadata. Its predictor columns are the source of truth for the model inputs: target-station engineered features plus raw measurements from every retained station at issue time `t`. The Ridge alpha search and validation policy are explicit constants so every fold and MLflow run remains inspectable.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. |
| `PREDICTION_PREVIEW_ROWS` | `5` | Number of scored test rows shown in the final preview. |
| `FULL_FEATURE_COLUMNS` | all metadata-declared predictors | The complete predictor contract used for common eligibility; raw timestamps and metadata fields are not model inputs. |
| `FEATURE_SUBSETS` | six predefined subsets | Candidate feature lists derived from the full metadata contract in metadata order. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` | The 24 future water levels predicted directly from one issue-time feature vector. |
| `FORECAST_HORIZON_HOURS` | `24` | Number of direct future target outputs and the metadata contract width. |
| `RIDGE_ALPHAS` | `[0.01, 0.1, 1.0, 10.0, 100.0]` | Candidate L2 regularization strengths. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `CV_SELECTION_METRIC` | `"mae"` | Aggregate CV metric used to select alpha; `"rmse"` is also supported. |
| `MLFLOW_EXPERIMENT_NAME` | `"ridge"` | Experiment receiving the parent, nested fold, and final test runs. |
| `MODEL_PATH` | `models/ridge_{TARGET_STATION_ID}.joblib` | Bundled scaler and selected Ridge estimator trained on all eligible training rows. |
| `MODEL_METADATA_PATH` | `models/ridge_{TARGET_STATION_ID}.json` | Reproducibility manifest containing the feature contract, selected subset and alpha, CV results, and training range. |

## Joint feature-subset and alpha search

The notebook compares one global `(feature subset, alpha)` pair with five expanding-window folds. The six predefined subsets are derived from metadata-declared predictors and retain their metadata order:

| Subset | Intended predictors | Current size |
| --- | --- | ---: |
| `full` | All declared predictors | 81 |
| `all_station_hydrology_quality_time` | Water-level history, imputation indicators, and calendar signals for every station; excludes weather | 55 |
| `raw_all_stations` | Current `water_level`, `imputed`, precipitation, and temperature for every station | 32 |
| `target_station_full` | All declared predictors for the target station only | 53 |
| `target_station_hydrology_quality_time` | Target-station water-level history, imputation indicators, and calendar signals | 41 |
| `current_water_levels_all_stations` | Current `water_level` for every station | 8 |

The full contract determines eligibility once for both artifacts. Consequently, all candidates use the same 48,403 training rows, 15,196 sealed-test rows, and identical fold indices in the current data. A missing predictor excluded by a candidate still removes that timestamp for every candidate; smaller subsets therefore do not gain additional eligible rows in this controlled ablation. The search performs `6 × 5 × 5 = 150` fold fits, then retrains only the selected candidate and evaluates the sealed test once.

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path
from uuid import uuid4

from joblib import dump
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.metrics import metric_tables
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
MLFLOW_EXPERIMENT_NAME = "ridge"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
feature_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
station_id = TARGET_STATION_ID
if station_id not in feature_metadata["engineered_station_ids"]:
    raise ValueError(
        f"Target station {station_id!r} is not engineered in the joined feature metadata"
    )
TARGET_VALID_COLUMN = f"{station_id}__target_valid"
TARGET_COLUMNS = [
    f"{station_id}__target_t_plus_{offset:02d}"
    for offset in range(1, FORECAST_HORIZON_HOURS + 1)
]
metadata_target_columns = feature_metadata.get("target_columns")
if feature_metadata.get("configuration", {}).get("horizon_hours") != FORECAST_HORIZON_HOURS:
    raise ValueError("Feature metadata horizon does not match FORECAST_HORIZON_HOURS")
if metadata_target_columns != TARGET_COLUMNS:
    raise ValueError("Feature metadata target columns do not match the configured horizon")
FULL_FEATURE_COLUMNS = list(feature_metadata["predictor_columns"])
if not FULL_FEATURE_COLUMNS:
    raise ValueError("The metadata predictor contract is empty")
# Backward-compatible alias for notebook consumers; eligibility uses the explicit
# full contract and candidate fits receive their selected columns explicitly.
FEATURE_COLUMNS = FULL_FEATURE_COLUMNS
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / f"ridge_{station_id}.joblib"
MODEL_METADATA_PATH = MODEL_DIR / f"ridge_{station_id}.json"

## Shared evaluation cohort

The model is fit and scored on rows from the joined feature artifacts. One row is one timestamp `t`, and it qualifies only when both conditions hold:

1. **Stage 3 marked the future window valid.** `{TARGET_STATION_ID}__target_valid` is true, and all 24 `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` values are present.
2. **Every model input is present.** All full-contract predictors must be available: the target station's engineered features plus every retained station's raw water level, imputation flag, precipitation, and temperature at issue time `t`.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it. Eligibility is deliberately based on `FULL_FEATURE_COLUMNS`, not a candidate subset, so all 30 candidates compare the same cohort.

## Helper functions

The eligibility helper validates the joined contract and returns rows with a valid 24-hour future window and complete predictors. The numeric conversion helper turns selected boolean/object predictors such as `imputed` columns into numeric values. Raw timestamps remain available for ordering and reporting but never enter the model matrix. The metric and preview helpers retain the direct 24-output order.

In [ ]:
def eligible_rows(frame: pd.DataFrame, *, station_id: str, artifact_name: str) -> pd.Series:
    """Return model-ready rows and reject incomplete joined artifacts."""
    required_columns = {
        "timestamp",
        TARGET_VALID_COLUMN,
        *FULL_FEATURE_COLUMNS,
        *TARGET_COLUMNS,
    }
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    target_valid_rows = frame[TARGET_VALID_COLUMN].eq(True)
    if frame.loc[target_valid_rows, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    eligible = (
        target_valid_rows
        & frame[FULL_FEATURE_COLUMNS].notna().all(axis=1)
        & frame[TARGET_COLUMNS].notna().all(axis=1)
    )
    return eligible


def _feature_parts(column: str) -> tuple[str, str]:
    """Split a joined predictor into station id and its declared base name."""
    station, separator, base_name = column.partition("__")
    if not separator or not station or not base_name:
        raise ValueError(
            f"Predictor {column!r} is not a joined station feature with '<station>__<name>' format"
        )
    return station, base_name


def _is_hydrology_quality_or_time(base_name: str) -> bool:
    return (
        base_name == "water_level"
        or base_name.startswith("water_level_")
        or base_name == "imputed"
        or base_name.startswith("imputed_count_")
        or base_name.startswith("utc_")
    )


def _is_weather(base_name: str) -> bool:
    return any(
        base_name == variable or base_name.startswith(f"{variable}_")
        for variable in WEATHER_VARIABLES
    )


def validate_feature_subsets(
    feature_subsets: dict[str, list[str]], full_feature_columns: list[str]
) -> None:
    """Validate candidate subsets against the full ordered predictor contract."""
    if not full_feature_columns:
        raise ValueError("The full predictor contract is empty")
    if len(full_feature_columns) != len(set(full_feature_columns)):
        raise ValueError("The full predictor contract contains duplicate columns")
    full_columns = set(full_feature_columns)
    for subset_name, subset_columns in feature_subsets.items():
        if not subset_columns:
            raise ValueError(f"Feature subset {subset_name!r} is empty")
        if len(subset_columns) != len(set(subset_columns)):
            raise ValueError(f"Feature subset {subset_name!r} contains duplicates")
        missing = sorted(set(subset_columns).difference(full_columns))
        if missing:
            raise ValueError(
                f"Feature subset {subset_name!r} contains undeclared columns: {missing}"
            )


def build_feature_subsets(
    full_feature_columns: list[str], target_station_id: str
) -> dict[str, list[str]]:
    """Derive the six ordered ablation subsets from metadata-declared predictors."""
    parsed_columns = [
        (column, *_feature_parts(column)) for column in full_feature_columns
    ]
    raw_names = {"water_level", "imputed", *WEATHER_VARIABLES}
    feature_subsets = {
        "full": list(full_feature_columns),
        "all_station_hydrology_quality_time": [
            column
            for column, station, base_name in parsed_columns
            if _is_hydrology_quality_or_time(base_name) and not _is_weather(base_name)
        ],
        "raw_all_stations": [
            column
            for column, station, base_name in parsed_columns
            if base_name in raw_names
        ],
        "target_station_full": [
            column
            for column, station, base_name in parsed_columns
            if station == target_station_id
        ],
        "target_station_hydrology_quality_time": [
            column
            for column, station, base_name in parsed_columns
            if station == target_station_id
            and _is_hydrology_quality_or_time(base_name)
        ],
        "current_water_levels_all_stations": [
            column
            for column, station, base_name in parsed_columns
            if base_name == "water_level"
        ],
    }
    validate_feature_subsets(feature_subsets, full_feature_columns)
    return feature_subsets


def numeric_predictors(
    frame: pd.DataFrame, feature_columns: list[str]
) -> pd.DataFrame:
    """Convert an explicit candidate feature list to numeric model inputs."""
    return frame[feature_columns].apply(pd.to_numeric, errors="raise").astype(float)


FEATURE_SUBSETS = build_feature_subsets(FULL_FEATURE_COLUMNS, station_id)


def time_series_splits(n_rows: int) -> tuple[TimeSeriesSplit, list[tuple[np.ndarray, np.ndarray]], int]:
    """Build and validate expanding chronological CV folds."""
    initial_train_rows = int(n_rows * INITIAL_TRAIN_FRACTION)
    validation_budget = n_rows - initial_train_rows - EMBARGO_HOURS
    validation_test_size = validation_budget // N_VALIDATION_FOLDS
    if validation_test_size < 1:
        raise ValueError("Not enough eligible training rows for the configured CV policy")

    splitter = TimeSeriesSplit(
        n_splits=N_VALIDATION_FOLDS,
        gap=EMBARGO_HOURS,
        test_size=validation_test_size,
    )
    splits = list(splitter.split(np.arange(n_rows)))
    if len(splits) != N_VALIDATION_FOLDS:
        raise ValueError(f"Expected {N_VALIDATION_FOLDS} validation folds, got {len(splits)}")

    previous_validation_end = -1
    for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
        splits, start=1
    ):
        if fold_train_indices.size == 0 or fold_validation_indices.size == 0:
            raise ValueError(f"Fold {fold_number} is empty")
        if not np.array_equal(fold_train_indices, np.arange(fold_train_indices.size)):
            raise ValueError(f"Fold {fold_number} training rows are not chronological")
        if fold_validation_indices[0] - fold_train_indices[-1] - 1 != EMBARGO_HOURS:
            raise ValueError(f"Fold {fold_number} does not have the configured embargo")
        if fold_validation_indices[0] <= previous_validation_end:
            raise ValueError("Validation folds overlap or are out of order")
        previous_validation_end = fold_validation_indices[-1]
    return splitter, splits, validation_test_size


def select_candidate(cv_results: pd.DataFrame, metric: str = "mae") -> tuple[str, float]:
    """Select one subset/alpha pair with deterministic tie-breaking."""
    if metric not in {"mae", "rmse"}:
        raise ValueError("CV selection metric must be either 'mae' or 'rmse'")
    metric_column = f"{metric}_mean"
    required_columns = {"subset", "alpha", "feature_count", metric_column}
    missing = sorted(required_columns.difference(cv_results.columns))
    if missing or cv_results.empty:
        raise ValueError(
            f"CV results are empty or missing columns: {missing}"
        )
    if cv_results[[metric_column, "alpha", "feature_count"]].isna().any().any():
        raise ValueError("CV candidate results contain null ranking values")
    ranked = cv_results.sort_values(
        [metric_column, "feature_count", "alpha", "subset"],
        kind="stable",
    )
    winner = ranked.iloc[0]
    return str(winner["subset"]), float(winner["alpha"])


# Descriptive alias used by callers that want to emphasize both dimensions.
select_feature_alpha = select_candidate

In [ ]:
def prediction_preview(
    frame: pd.DataFrame, predictions: np.ndarray
) -> pd.DataFrame:
    """Return issue timestamps, 24 actual targets, and direct predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)


def predicted_vs_actual_figure(
    actual: pd.DataFrame,
    predictions: np.ndarray,
    target_columns: list[str],
    *,
    title: str = "Ridge predicted vs actual",
) -> plt.Figure:
    """Build a 5x5 actual-versus-predicted scatterplot grid."""
    alpha = 0.35
    actual_values = actual[target_columns].to_numpy(dtype=float)
    prediction_values = np.asarray(predictions, dtype=float)
    if prediction_values.shape != actual_values.shape:
        raise ValueError(
            "Prediction shape must match actual target shape: "
            f"{prediction_values.shape} != {actual_values.shape}"
        )

    all_values = np.concatenate((actual_values.ravel(), prediction_values.ravel()))
    axis_min = float(np.min(all_values))
    axis_max = float(np.max(all_values))
    padding = max((axis_max - axis_min) * 0.05, 1e-9)
    axis_min -= padding
    axis_max += padding

    fig, axes = plt.subplots(5, 5, figsize=(20, 20), sharex=True, sharey=True)
    axes = axes.ravel()
    for horizon, target_column in enumerate(target_columns):
        axes[horizon].scatter(
            actual_values[:, horizon],
            prediction_values[:, horizon],
            s=12,
            alpha=alpha,
        )
        axes[horizon].set_title(target_column.rsplit("__", maxsplit=1)[-1])

    combined_axis = axes[len(target_columns)]
    combined_axis.scatter(
        actual_values.ravel(),
        prediction_values.ravel(),
        s=12,
        alpha=alpha,
    )
    combined_axis.set_title("All horizons")

    for axis in axes:
        axis.plot(
            [axis_min, axis_max],
            [axis_min, axis_max],
            color="black",
            linestyle="--",
            linewidth=1,
        )
        axis.set_xlim(axis_min, axis_max)
        axis.set_ylim(axis_min, axis_max)
        axis.grid(alpha=0.25)
    fig.supxlabel("Actual values")
    fig.supylabel("Predicted values")
    fig.suptitle(title)
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.97))
    return fig

In [ ]:
def error_boxplots_figure(
    boxplot_values: dict[str, list[np.ndarray]],
    category_labels: list[str],
    *,
    title: str,
    x_axis_label: str = "Forecast horizon",
    summary_markers: dict[str, np.ndarray] | None = None,
) -> plt.Figure:
    """Build horizon-wise box plots with optional summary markers."""
    positions = list(range(1, len(category_labels) + 1))
    if not boxplot_values:
        raise ValueError("At least one box-plot series is required")
    for label, values in boxplot_values.items():
        if len(values) != len(positions):
            raise ValueError(
                f"{label} does not cover every category"
            )
        if any(
            values_for_horizon.size == 0
            or not np.isfinite(values_for_horizon).all()
            for values_for_horizon in map(np.asarray, values)
        ):
            raise ValueError(f"{label} contains empty or non-finite values")
    if summary_markers is not None:
        for label, values in summary_markers.items():
            values = np.asarray(values, dtype=float)
            if values.shape != (len(positions),) or not np.isfinite(values).all():
                raise ValueError(
                    f"{label} markers do not cover every category with finite values"
                )

    fig, axis = plt.subplots(figsize=(20, 6))
    series_count = len(boxplot_values)
    group_width = 0.8
    box_width = group_width / series_count * 0.8
    offsets = (
        np.arange(series_count) - (series_count - 1) / 2
    ) * (group_width / series_count)
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    for series_number, (label, values) in enumerate(boxplot_values.items()):
        color = colors[series_number % len(colors)]
        boxplot = axis.boxplot(
            values,
            positions=np.asarray(positions) + offsets[series_number],
            widths=box_width,
            patch_artist=True,
            boxprops={"facecolor": color, "alpha": 0.35},
            medianprops={"color": color, "linewidth": 1.5},
        )
        boxplot["boxes"][0].set_label(label)
        if summary_markers is not None and label in summary_markers:
            axis.plot(
                np.asarray(positions) + offsets[series_number],
                summary_markers[label],
                color=color,
                marker="D",
                linestyle="none",
                label=f"{label} summary",
            )
    axis.set_ylabel("Error")
    axis.set_xticks(positions)
    axis.set_xticklabels(category_labels)
    axis.set_xlabel(x_axis_label)
    axis.legend()
    axis.grid(axis="y", alpha=0.25)
    fig.suptitle(title)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    return fig


def horizon_error_boxplot_payload(
    actual: pd.DataFrame,
    predictions: np.ndarray,
    per_horizon_metrics: pd.DataFrame,
    target_columns: list[str],
) -> tuple[dict[str, list[np.ndarray]], list[str], dict[str, np.ndarray]]:
    """Build per-horizon test-error box values and summary markers."""
    actual_values = actual[target_columns].to_numpy(dtype=float)
    prediction_values = np.asarray(predictions, dtype=float)
    if prediction_values.shape != actual_values.shape:
        raise ValueError(
            "Prediction shape must match actual target shape: "
            f"{prediction_values.shape} != {actual_values.shape}"
        )
    expected_horizons = np.arange(1, len(target_columns) + 1)
    if not np.array_equal(
        per_horizon_metrics["horizon_hours"].to_numpy(), expected_horizons
    ):
        raise ValueError("Horizon metrics are not in forecast-horizon order")

    absolute_errors = np.abs(actual_values - prediction_values)
    if not np.isfinite(absolute_errors).all():
        raise ValueError("Test errors contain non-finite values")
    horizon_labels = [f"H+{horizon:02d}" for horizon in expected_horizons]
    boxplot_values = {
        metric.upper(): [
            absolute_errors[:, horizon - 1] for horizon in expected_horizons
        ]
        for metric in ("mae", "rmse")
    }
    summary_markers = {
        metric.upper(): per_horizon_metrics[metric].to_numpy(dtype=float)
        for metric in ("mae", "rmse")
    }
    return boxplot_values, horizon_labels, summary_markers

## Load joined feature artifacts

Loads `all_stations_train_features.parquet` and `all_stations_test_features.parquet` from the joined Stage-3 directory. The metadata was loaded during setup and is checked before the fit, so a missing or incompatible artifact fails before any model work begins.

In [ ]:
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
for artifact_path in (METADATA_PATH, train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(f"Missing joined feature artifact: {artifact_path}")

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)

## Apply the eligibility cohort

Builds the train and test masks with `eligible_rows()` and keeps only rows that pass. If either split has no eligible row, the notebook stops rather than fitting on an empty frame or reporting a metric computed from nothing.

In [ ]:
train_mask = eligible_rows(
    train_features, station_id=station_id, artifact_name="train"
)
test_mask = eligible_rows(
    test_features, station_id=station_id, artifact_name="test"
)
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

train_rows = (
    train_features.loc[train_mask]
    .sort_values("timestamp", kind="mergesort")
    .reset_index(drop=True)
)
test_rows = (
    test_features.loc[test_mask]
    .sort_values("timestamp", kind="mergesort")
    .reset_index(drop=True)
)
if not train_rows["timestamp"].is_monotonic_increasing:
    raise ValueError("Eligible training rows are not chronological")

## Joint time-series subset and alpha search

Eligible training rows are sorted by issue time before `TimeSeriesSplit` creates five expanding-window folds. The explicit `test_size` allocates the post-initial-training portion across the folds, while the 24-row gap acts as the requested hourly embargo. Each `(subset, alpha)` candidate has one MLflow parent and each fold has one nested child run: `6 × 5 × 5 = 150` fits. The current execution must produce the complete 30-candidate Cartesian product before selection.

Every fold fits its own `StandardScaler` and 24-output `Ridge` model using only that fold's training rows and the candidate's explicit columns. The sealed test cohort is not referenced until the final fit below.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splitter, cv_splits, validation_test_size = time_series_splits(len(train_rows))
cv_results_rows = []
cv_horizon_rows_by_candidate = {}
expected_candidate_keys = {
    (subset_name, float(alpha))
    for subset_name in FEATURE_SUBSETS
    for alpha in RIDGE_ALPHAS
}

for subset_name, feature_columns in FEATURE_SUBSETS.items():
    for alpha in RIDGE_ALPHAS:
        fold_aggregate_rows = []
        fold_horizon_rows = []
        with mlflow.start_run(
            run_name=f"ridge_cv_{subset_name}_{alpha:g}",
            nested=False,
            tags={
                "phase": "cv",
                "run_type": "candidate_parent",
                "subset": subset_name,
                "execution_uuid": NOTEBOOK_EXECUTION_UUID,
            },
        ):
            mlflow.log_params({
                "phase": "cv",
                "run_type": "candidate_parent",
                "subset": subset_name,
                "feature_count": len(feature_columns),
                "feature_columns": json.dumps(feature_columns),
                "alpha": alpha,
                "n_validation_folds": N_VALIDATION_FOLDS,
                "validation_test_size": validation_test_size,
                "embargo_hours": EMBARGO_HOURS,
                "selection_metric": CV_SELECTION_METRIC,
                "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                "common_train_rows": len(train_rows),
            })

            for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
                cv_splits, start=1
            ):
                fold_train_rows = train_rows.iloc[fold_train_indices]
                fold_validation_rows = train_rows.iloc[fold_validation_indices]
                fold_scaler = StandardScaler()
                fold_train_predictors = fold_scaler.fit_transform(
                    numeric_predictors(fold_train_rows, feature_columns)
                )
                fold_validation_predictors = fold_scaler.transform(
                    numeric_predictors(fold_validation_rows, feature_columns)
                )
                fold_ridge = Ridge(alpha=alpha)
                fold_ridge.fit(fold_train_predictors, fold_train_rows[TARGET_COLUMNS])
                fold_predictions = np.asarray(
                    fold_ridge.predict(fold_validation_predictors)
                )
                if fold_predictions.shape != (len(fold_validation_rows), len(TARGET_COLUMNS)):
                    raise ValueError(f"Unexpected fold prediction shape: {fold_predictions.shape}")
                if not np.isfinite(fold_predictions).all():
                    raise ValueError("Ridge produced non-finite fold predictions")

                fold_aggregate, fold_per_horizon = metric_tables(
                    fold_validation_rows[TARGET_COLUMNS],
                    fold_predictions,
                    target_columns=TARGET_COLUMNS,
                    station_id=station_id,
                )
                fold_aggregate_rows.append(fold_aggregate.iloc[0])
                fold_horizon_rows.append(fold_per_horizon)
                with mlflow.start_run(
                    run_name=f"ridge_cv_{subset_name}_{alpha:g}_fold_{fold_number}",
                    nested=True,
                    tags={
                        "phase": "cv",
                        "run_type": "fold",
                        "subset": subset_name,
                        "fold": str(fold_number),
                        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                    },
                ):
                    mlflow.log_params({
                        "phase": "cv",
                        "run_type": "fold",
                        "subset": subset_name,
                        "feature_count": len(feature_columns),
                        "alpha": alpha,
                        "fold": fold_number,
                        "train_rows": len(fold_train_rows),
                        "validation_rows": len(fold_validation_rows),
                        "gap_rows": EMBARGO_HOURS,
                        "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                        "train_start": fold_train_rows["timestamp"].iloc[0].isoformat(),
                        "train_end": fold_train_rows["timestamp"].iloc[-1].isoformat(),
                        "validation_start": fold_validation_rows["timestamp"].iloc[0].isoformat(),
                        "validation_end": fold_validation_rows["timestamp"].iloc[-1].isoformat(),
                        "train_index_start": int(fold_train_indices[0]),
                        "train_index_end": int(fold_train_indices[-1]),
                        "validation_index_start": int(fold_validation_indices[0]),
                        "validation_index_end": int(fold_validation_indices[-1]),
                    })
                    mlflow.log_metrics({
                        "mae": float(fold_aggregate.iloc[0]["mae"]),
                        "rmse": float(fold_aggregate.iloc[0]["rmse"]),
                        **{
                            f"mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                            for row in fold_per_horizon.itertuples()
                        },
                        **{
                            f"rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                            for row in fold_per_horizon.itertuples()
                        },
                    })

            fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
            fold_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
            parent_metrics = {
                "cv_mae_mean": float(fold_aggregate_metrics["mae"].mean()),
                "cv_mae_std": float(fold_aggregate_metrics["mae"].std(ddof=0)),
                "cv_rmse_mean": float(fold_aggregate_metrics["rmse"].mean()),
                "cv_rmse_std": float(fold_aggregate_metrics["rmse"].std(ddof=0)),
            }
            for horizon, horizon_metrics in fold_horizon_metrics.groupby("horizon_hours"):
                parent_metrics[f"cv_mae_horizon_{horizon:02d}_mean"] = float(horizon_metrics["mae"].mean())
                parent_metrics[f"cv_mae_horizon_{horizon:02d}_std"] = float(horizon_metrics["mae"].std(ddof=0))
                parent_metrics[f"cv_rmse_horizon_{horizon:02d}_mean"] = float(horizon_metrics["rmse"].mean())
                parent_metrics[f"cv_rmse_horizon_{horizon:02d}_std"] = float(horizon_metrics["rmse"].std(ddof=0))
            candidate_key = (subset_name, float(alpha))
            cv_horizon_rows_by_candidate[candidate_key] = fold_horizon_rows.copy()
            mlflow.log_metrics(parent_metrics)
            cv_results_rows.append({
                "subset": subset_name,
                "feature_count": len(feature_columns),
                "alpha": float(alpha),
                "mae_mean": parent_metrics["cv_mae_mean"],
                "mae_std": parent_metrics["cv_mae_std"],
                "rmse_mean": parent_metrics["cv_rmse_mean"],
                "rmse_std": parent_metrics["cv_rmse_std"],
                **{
                    metric_name: metric_value
                    for metric_name, metric_value in parent_metrics.items()
                    if metric_name not in {"cv_mae_mean", "cv_mae_std", "cv_rmse_mean", "cv_rmse_std"}
                },
            })

cv_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if cv_experiment is None:
    raise ValueError(f"MLflow experiment {MLFLOW_EXPERIMENT_NAME!r} was not found")
current_cv_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'candidate_parent'"
    ),
)
current_fold_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'fold'"
    ),
)
parent_keys = {
    (str(row["tags.subset"]), float(row["params.alpha"]))
    for _, row in current_cv_runs.iterrows()
}
if len(current_cv_runs) != len(expected_candidate_keys) or parent_keys != expected_candidate_keys:
    raise ValueError(
        "Current execution must produce the complete 30-candidate subset/alpha product: "
        f"expected {len(expected_candidate_keys)} {sorted(expected_candidate_keys)}, "
        f"got {len(current_cv_runs)} {sorted(parent_keys)}"
    )
expected_fold_count = len(expected_candidate_keys) * N_VALIDATION_FOLDS
if len(current_fold_runs) != expected_fold_count:
    raise ValueError(
        f"Current execution must produce {expected_fold_count} nested fold runs, got {len(current_fold_runs)}"
    )
fold_keys = {
    (
        str(row["tags.subset"]),
        float(row["params.alpha"]),
        int(row["tags.fold"]),
    )
    for _, row in current_fold_runs.iterrows()
}
expected_fold_keys = {
    (subset_name, float(alpha), fold_number)
    for subset_name, alpha in expected_candidate_keys
    for fold_number in range(1, N_VALIDATION_FOLDS + 1)
}
if fold_keys != expected_fold_keys:
    raise ValueError("Current execution fold runs do not cover every candidate and fold")
cv_results = pd.DataFrame(cv_results_rows)
if len(cv_results) != len(expected_candidate_keys):
    raise ValueError("The in-memory CV result table is incomplete")
if set(zip(cv_results["subset"], cv_results["alpha"])) != expected_candidate_keys:
    raise ValueError("The in-memory CV result table does not match the candidate product")
cv_results = cv_results.sort_values(["subset", "alpha"], kind="stable").reset_index(drop=True)
selected_subset, selected_alpha = select_candidate(cv_results, CV_SELECTION_METRIC)
selected_feature_columns = FEATURE_SUBSETS[selected_subset]
fold_horizon_rows = cv_horizon_rows_by_candidate[(selected_subset, selected_alpha)]
print(
    f"Selected Ridge candidate by CV {CV_SELECTION_METRIC.upper()}: "
    f"{selected_subset!r}, alpha={selected_alpha:g}"
)
display(cv_results[["subset", "feature_count", "alpha", "mae_mean", "mae_std", "rmse_mean", "rmse_std"]])

## Retrain the selected subset and alpha

The selected `(feature subset, alpha)` pair is retrained once on all eligible, chronologically ordered training rows. The scaler and Ridge estimator are bundled in a single pipeline, fitted on the full eligible training cohort, and persisted with a reproducibility manifest before the sealed test predictors are scored.

In [ ]:
final_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=selected_alpha)),
    ]
)
final_model.fit(
    numeric_predictors(train_rows, selected_feature_columns),
    train_rows[TARGET_COLUMNS],
)
test_predictions = np.asarray(
    final_model.predict(numeric_predictors(test_rows, selected_feature_columns))
)
if test_predictions.shape != (len(test_rows), len(TARGET_COLUMNS)):
    raise ValueError(f"Unexpected prediction shape: {test_predictions.shape}")
if not np.isfinite(test_predictions).all():
    raise ValueError("Ridge produced non-finite predictions")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
dump(final_model, MODEL_PATH)
cv_results_records = []
for row in cv_results.to_dict(orient="records"):
    cv_results_records.append({
        key: (
            str(value)
            if key == "subset"
            else int(value)
            if key == "feature_count"
            else float(value)
        )
        for key, value in row.items()
    })
model_manifest = {
    "schema_version": "1.1",
    "model_path": str(MODEL_PATH),
    "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    "model_type": "sklearn.pipeline.Pipeline",
    "estimator": "Ridge",
    "preprocessor": "StandardScaler",
    "station_id": station_id,
    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
    "full_feature_columns": FULL_FEATURE_COLUMNS,
    "selected_subset": selected_subset,
    "feature_subset": selected_subset,
    "selected_feature_columns": selected_feature_columns,
    "feature_subsets": FEATURE_SUBSETS,
    "target_columns": TARGET_COLUMNS,
    "selected_alpha": float(selected_alpha),
    "selection_metric": CV_SELECTION_METRIC,
    "tie_breaking": [
        f"lowest aggregate CV {CV_SELECTION_METRIC.upper()}",
        "fewer features",
        "smaller alpha",
        "stable subset name",
    ],
    "cv_results": cv_results_records,
    "common_cohort_eligibility": {
        "contract": "full_feature_columns",
        "rule": "target_valid and complete full predictor and target contract",
        "train_raw_rows": int(len(train_features)),
        "train_eligible_rows": int(len(train_rows)),
        "test_raw_rows": int(len(test_features)),
        "test_eligible_rows": int(len(test_rows)),
        "same_folds_for_all_candidates": True,
    },
    "training": {
        "source_artifact": str(train_path),
        "raw_rows": int(len(train_features)),
        "eligible_rows": int(len(train_rows)),
        "eligibility": "target_valid and complete full predictor and target contract",
        "timestamp_start": train_rows["timestamp"].iloc[0].isoformat(),
        "timestamp_end": train_rows["timestamp"].iloc[-1].isoformat(),
    },
}
MODEL_METADATA_PATH.write_text(
    json.dumps(model_manifest, indent=2) + "\n", encoding="utf-8"
)
print(f"Saved Ridge model to {MODEL_PATH}")
print(f"Saved Ridge model manifest to {MODEL_METADATA_PATH}")

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort reports aggregate MAE/RMSE, the same metrics for each lead in the direct 24-hour forecast, and a short preview for comparison with actual targets. Plot and MLflow labels identify both the selected subset and alpha. There is no second pass and no refitting.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
if not np.isfinite(aggregate_metrics[["mae", "rmse"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite horizon metrics")
with mlflow.start_run(
    run_name=f"ridge_test_{selected_subset}_alpha_{selected_alpha:g}",
    nested=False,
    tags={"phase": "test", "run_type": "sealed_test", "subset": selected_subset, "execution_uuid": NOTEBOOK_EXECUTION_UUID},
):
    mlflow.log_params({
        "phase": "test",
        "run_type": "sealed_test",
        "subset": selected_subset,
        "feature_count": len(selected_feature_columns),
        "feature_columns": json.dumps(selected_feature_columns),
        "alpha": selected_alpha,
        "selection_metric": CV_SELECTION_METRIC,
        "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
        "cv_selected_metric": float(cv_results.loc[
            cv_results["subset"].eq(selected_subset)
            & cv_results["alpha"].eq(selected_alpha),
            f"{CV_SELECTION_METRIC}_mean",
        ].iloc[0]),
        "scored_issue_times": len(test_rows),
    })
    mlflow.log_metrics({
        "mae": float(aggregate_metrics.iloc[0]["mae"]),
        "rmse": float(aggregate_metrics.iloc[0]["rmse"]),
        **{
            f"mae_horizon_{row.horizon_hours:02d}": float(row.mae)
            for row in per_horizon_metrics.itertuples()
        },
        **{
            f"rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
            for row in per_horizon_metrics.itertuples()
        },
    })
    cv_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
    horizons = list(range(1, len(TARGET_COLUMNS) + 1))
    horizon_labels = [f"H+{horizon:02d}" for horizon in horizons]
    cv_boxplot_values = {
        metric.upper(): [
            cv_horizon_metrics.loc[
                cv_horizon_metrics["horizon_hours"].eq(horizon), metric
            ].to_numpy(dtype=float)
            for horizon in horizons
        ]
        for metric in ("mae", "rmse")
    }
    cv_rmse_mae_boxplots_fig = error_boxplots_figure(
        cv_boxplot_values,
        horizon_labels,
        title=f"Ridge CV errors — {selected_subset}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(cv_rmse_mae_boxplots_fig, "cv_rmse_mae_boxplots.png")
    plt.show()
    plt.close(cv_rmse_mae_boxplots_fig)
    (
        test_boxplot_values,
        test_horizon_labels,
        test_summary_markers,
    ) = horizon_error_boxplot_payload(
        test_rows,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
    )
    test_error_boxplots_fig = error_boxplots_figure(
        test_boxplot_values,
        test_horizon_labels,
        title=f"Ridge final-test errors — {selected_subset}, alpha={selected_alpha:g}",
        x_axis_label="Forecast horizon",
        summary_markers=test_summary_markers,
    )
    mlflow.log_figure(test_error_boxplots_fig, "test_error_boxplots.png")
    plt.show()
    plt.close(test_error_boxplots_fig)
    test_predicted_vs_actual_fig = predicted_vs_actual_figure(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title=f"Ridge predicted vs actual — {selected_subset}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(test_predicted_vs_actual_fig, "test_predicted_vs_actual.png")
    plt.show()
    plt.close(test_predicted_vs_actual_fig)
print(
    f"Ridge test results for {station_id} "
    f"(selected subset={selected_subset!r}, alpha={selected_alpha:g})"
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))